In [2]:
import json
import pandas as pd
import os 
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

data = []
base_path = os.path.dirname(os.getcwd())
file_path = os.path.join(base_path, "log", "cowrie.json")

with open(file_path) as f:
    data = [json.loads(line) for line in f]
    
df = pd.DataFrame(data)
pd.set_option('display.max_columns', None)
print(df.head(1)) #1 evento -> 1 fila

                  eventid         src_ip  src_port         dst_ip  dst_port  \
0  cowrie.session.connect  101.126.41.73   58982.0  172.31.45.231    2222.0   

        session protocol  \
0  8fb6f3d375c6      ssh   

                                                                            message  \
0  New connection: 101.126.41.73:58982 (172.31.45.231:2222) [session: 8fb6f3d375c6]   

             sensor                                  uuid  \
0  ip-172-31-45-231  4a689bfa-1c75-11f1-aaec-e71796b4ff8b   

                     timestamp version hassh hasshAlgorithms kexAlgs keyAlgs  \
0  2026-04-07T07:55:30.885204Z     NaN   NaN             NaN     NaN     NaN   

  encCS macCS compCS langCS duration username password arch input ttylog  \
0   NaN   NaN    NaN    NaN      NaN      NaN      NaN  NaN   NaN    NaN   

   size shasum duplicate outfile destfile  
0   NaN    NaN       NaN     NaN      NaN  


In [3]:
sessions = []
print(df.columns)
grouped = df.groupby("session") #Agrupas el df por sesión
for session_id, group in grouped:
    #Diccionario de esta sesion
    session_data = {}

    #Guarda ID de la sesión
    session_data["session"] = session_id

        # IP atacante
    session_data["src_ip"] = group["src_ip"].dropna().iloc[0] if not group["src_ip"].dropna().empty else None

    # Puerto origen 
    session_data["src_port"] = group["src_port"].dropna().iloc[0] if not group["src_port"].dropna().empty else None

        # Duración segun evento cierre
    duration = group[group["eventid"] == "cowrie.session.closed"]["duration"]
    session_data["duration"] = float(duration.iloc[0]) if not duration.empty else 0

    # Login exitoso (1 = si, 0 = no)
    session_data["login_success"] = int((group["eventid"] == "cowrie.login.success").any())

    # Intentos de login fallidos
    session_data["login_attempts"] = (group["eventid"] == "cowrie.login.failed").sum()

    # Nº comandos ejecutados por el atacante 
    session_data["num_commands"] = (group["eventid"] == "cowrie.command.input").sum()

    # Descarga de archivos en la sesión (1 = sí, 0 = no)
    session_data["file_download"] = int((group["eventid"] == "cowrie.session.file_download").any())

    #Comandos atacante
    commands = group[group["eventid"] == "cowrie.command.input"]["input"].dropna().tolist()
    session_data["commands"] = commands

    #Distintos ataques en base a comandos
    # 1. Reconocimiento
    session_data["recon"] = int(
        any(any(x in cmd.lower() for x in ["uname","whoami","id","ifconfig","pwd"]) for cmd in commands)
    )

    # 2. Descarga malware
    session_data["download"] = int(
        any(any(x in cmd.lower() for x in ["wget","curl","tftp"]) for cmd in commands)
    )

    # 3. Cambio permisos
    session_data["chmod"] = int(
        any("chmod" in cmd.lower() for cmd in commands)
    )

    # 4. Ejecución de binarios/scripts
    session_data["execution"] = int(
        any(any(x in cmd.lower() for x in ["./","sh ","bash "]) for cmd in commands)
    )

    # 5. Persistencia
    session_data["persistence"] = int(
        any(".ssh" in cmd.lower() or "authorized_keys" in cmd.lower() for cmd in commands)
    )

    #Variedad de comandos en el ataque
    session_data["unique_commands"] = len(set(commands)) 

    #Complejidad del comando
    if commands:
        avg_len = sum(map(len, commands)) / len(commands)
    else:
        avg_len = 0
    session_data["avg_command_length"] = avg_len

    # Velocidad del ataque, alto->bot, bajo -> humano
    session_data["commands_per_second"] = (
        session_data["num_commands"] / session_data["duration"]
        if session_data["duration"] > 0 else 0
    )

    # Agrega diccionario de sesión
    sessions.append(session_data)
    
# Crea dataframe a partir de todos los diccionarios
dataset = pd.DataFrame(sessions)

print(dataset.head(50))

Index(['eventid', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'session',
       'protocol', 'message', 'sensor', 'uuid', 'timestamp', 'version',
       'hassh', 'hasshAlgorithms', 'kexAlgs', 'keyAlgs', 'encCS', 'macCS',
       'compCS', 'langCS', 'duration', 'username', 'password', 'arch', 'input',
       'ttylog', 'size', 'shasum', 'duplicate', 'outfile', 'destfile'],
      dtype='str')
         session           src_ip  src_port  duration  login_success  \
0   0562de4fe30a    45.194.37.246   43234.0       4.5              0   
1   0587c101b68a    101.126.41.73   45656.0       2.3              0   
2   0695e64a2061    101.126.41.73   42906.0     120.0              0   
3   08219aebc679    45.194.37.246   47468.0       4.1              0   
4   08ea45147473    45.194.37.246   59112.0       2.3              0   
5   0b9a6213e7cb    101.126.41.73   37884.0       5.6              1   
6   0de1431c7020    101.126.41.73   37896.0      25.5              0   
7   125d14836e72    101.126.41.73

In [14]:
# Cuenta de cuantas veces nos ha atacado cada IP.
group_by_ip = dataset["src_ip"].value_counts()
print(group_by_ip)




src_ip
101.126.41.73      35
45.194.37.246      28
102.210.149.130    21
103.236.140.19      7
103.59.94.117       3
175.27.145.234      1
106.12.157.104      1
Name: count, dtype: int64


In [17]:
import geoip2.database

# Abrir la base de datos local
dic = {}
reader = geoip2.database.Reader('../db/GeoLite2-City.mmdb')
for i,k in  group_by_ip.items():

    response = reader.city(i)
    if response.country.name not in dic.keys():
        dic[response.country.name] = k
    else:
        dic[response.country.name] += k

reader.close()

print(dic)


{'China': 37, 'Seychelles': 28, 'South Africa': 21, 'Indonesia': 10}


In [59]:
# Comandos ejecutados

dataset_filtrado = dataset[dataset["commands"].str.len() > 0]
print(dataset_filtrado["commands"])

5                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     [cd ~; chattr -ia .ssh; lockr -ia .ssh, cd ~ && rm -rf .ssh && mkdir .ssh && echo "ssh-rsa AAAAB3NzaC1yc2EAAAABJQAAAQEArDp4cun2lhr4KUhBGE7VvAcwdli2a8dbnrTOrbMz1+5O73fcBOx8NVbUT0bUanUV9tJ2/9p7+vD0EpZ3Tz/+0kX34uAx1RV/75GVOmNx+9EuWOnvNoaJe0QXxziIg9eLBHpgLMuakb5+BgTFB+rKJAw9u9FSTDengvS8hX1kNFS4Mjux0hJOK8rvcEmPecjdySYMb66nylAKGwCEE6WEQHmd1mUPgHwGQ0hWCwsQk13yCGPK5w6hYp5zYkFnvlC8hGmd4Ww+u97k6pfTGTUbJk14ujvcD9iUKQTTWYYjIIu5PmUux5bsZ0R4WFwdIe6+i6rBLAsPKgA

In [20]:
print(dataset["duration"])
low = 0
med = 0
high = 0
for i in dataset["duration"]:
    if i < 20:
        low += 1
    if 20<= i < 80:
        med += 1
    if i >= 80:
        high += 1
print("Low duration connections: ", low)
print("Medium duration connections: ", med)
print("High duration connections: ", high)




0       4.5
1       2.3
2     120.0
3       4.1
4       2.3
5       5.6
6      25.5
7      63.1
8       6.2
9       4.4
10      2.3
11     20.0
12      1.1
13      3.1
14    120.0
15      4.7
16      3.2
17      4.2
18      2.2
19      2.2
20      4.0
21      4.2
22    120.0
23      2.3
24    120.0
25      1.3
26      2.2
27      4.3
28      4.1
29      1.2
30    120.0
31      3.7
32     24.9
33      2.4
34      4.1
35      4.5
36      1.5
37      6.0
38    120.0
39      4.3
40      2.2
41      4.3
42      1.1
43    138.5
44      2.3
45      4.3
46      6.2
47      4.0
48     14.8
49      2.2
50      3.8
51      2.2
52    120.0
53    120.0
54    120.0
55    120.0
56      3.9
57      2.2
58    120.0
59      6.1
60     12.6
61    120.1
62      4.3
63      4.3
64      6.0
65      2.5
66     15.7
67      1.3
68      2.2
69      1.1
70      2.2
71      2.2
72      2.1
73      2.4
74     14.2
75      2.2
76      2.2
77      4.0
78      1.5
79    120.0
80    150.1
81    120.0
82      6.0
83  

In [ ]:
si = 0
no = 0
for i in dataset["chmod"]:
    if i != 0:
        si +=1
    else:
        no += 1

print("Numero de permisos cambiados", si)
print("Numero de permisos no cambiados", no)